# Simple Linear Regression — Deep Learning Notebook
This notebook preserves the full learning path: **theory → intuition → arithmetic → Python → visualization → sklearn → domain transfer → exercises**.

Learning rule: **understand → implement → visualize → use library → apply → explain back.**


## 1. 📘 Theory
Simple linear regression predicts a **continuous numerical target** from **one input feature**.

\[
\hat{Y}=mX+c
\]

- `X` = feature/input/predictor
- `Y` = actual target
- `Ŷ` = predicted target
- `m` = slope
- `c` = intercept
- `Y-Ŷ` = residual

Example mental model: **Rainfall → Crop Yield**. Regression learns association; it does not automatically prove causation.


## 2. 🧠 Intuition
Imagine putting a ruler through scattered points. There are infinitely many possible lines, so we need a definition of **best**.

Ordinary least squares chooses the line that minimizes the total **squared prediction error**.


## 3. 🔢 Geometry
For `y = mx + c`:
- `m` tells how much predicted `Y` changes for a +1 change in `X`.
- `c` is the predicted `Y` when `X = 0`.

If `m=0.6`, each +1 in X changes predicted Y by about +0.6. If `c=2.2`, the line crosses the Y-axis at 2.2.


## 4. 💻 Training data and row pairing
Dataset from the course:

| X | Y |
|---:|---:|
|1|2|
|2|4|
|3|5|
|4|4|
|5|5|

`x[i]` and `y[i]` form **one observation**. This is why one shared index is correct; nested loops would create every X/Y combination instead of five paired observations.


In [ ]:
x = [1, 2, 3, 4, 5]
y = [2, 4, 5, 4, 5]

for i in range(len(x)):
    print("row", i, "X =", x[i], "Y =", y[i])


## 5. 🔢 Mean
\[
\bar{x}=\frac{\sum x_i}{n},\qquad \bar{y}=\frac{\sum y_i}{n}
\]

For this dataset:
\[
\bar{x}=3,\qquad \bar{y}=4
\]

The sigma `Σ` can be mentally translated into **calculate a row-level quantity and keep adding it**.


In [ ]:
x_total = 0
y_total = 0
count = len(x)

for i in range(count):
    x_total = x_total + x[i]
    y_total = y_total + y[i]

x_mean = x_total / count
y_mean = y_total / count

print("x_mean =", x_mean)
print("y_mean =", y_mean)


## 6. 🧠 Deviations from the mean
\[
x_{diff}=x_i-\bar{x},\qquad y_{diff}=y_i-\bar{y}
\]

| X | Y | X-X̄ | Y-Ȳ |
|---:|---:|---:|---:|
|1|2|-2|-2|
|2|4|-1|0|
|3|5|0|1|
|4|4|1|0|
|5|5|2|1|

A deviation says how far a value is from its dataset average.


In [ ]:
for i in range(count):
    x_diff = x[i] - x_mean
    y_diff = y[i] - y_mean
    print(x[i], y[i], x_diff, y_diff)


## 7. 🔢 Slope from scratch
Use the centered/deviation form:

\[
m=\frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{\sum(x_i-\bar{x})^2}
\]

Mental model:

\[
\text{slope}=\frac{\text{how X and Y move together}}{\text{how much X moves}}
\]

Important: **do not divide row by row**. Accumulate all numerator pieces, accumulate all denominator pieces, then divide once.


In [ ]:
numerator = 0
denominator = 0

for i in range(count):
    x_diff = x[i] - x_mean
    y_diff = y[i] - y_mean

    xy_diff = x_diff * y_diff
    x_diff_squared = x_diff ** 2

    numerator = numerator + xy_diff
    denominator = denominator + x_diff_squared

    print("x_diff =", x_diff,
          "y_diff =", y_diff,
          "xy_diff =", xy_diff,
          "x_diff² =", x_diff_squared)

m = numerator / denominator

print("numerator =", numerator)       # 6
print("denominator =", denominator)   # 10
print("m =", m)                       # 0.6


### Arithmetic
| Xdiff | Ydiff | Xdiff×Ydiff | Xdiff² |
|---:|---:|---:|---:|
|-2|-2|4|4|
|-1|0|0|1|
|0|1|0|0|
|1|0|0|1|
|2|1|2|4|
| | |**6**|**10**|

So:
\[
m=6/10=0.6
\]

The larger course formula is algebraically equivalent. Understanding this version is more valuable than memorizing both.


## 8. 🔢 Intercept
\[
c=\bar{y}-m\bar{x}
\]

\[
c=4-(0.6)(3)=2.2
\]

So the learned model is:
\[
\boxed{\hat{Y}=0.6X+2.2}
\]

For ordinary least-squares regression with an intercept, the line passes through \((\bar{x},\bar{y})\).


In [ ]:
c = y_mean - (m * x_mean)
print("c =", c)


## 9. 💻 Prediction
Prediction happens **after** the parameters are learned:
\[
\hat{Y}=mX+c
\]


In [ ]:
def predict(x_value, m, c):
    return (m * x_value) + c

for x_value in x:
    print("X =", x_value, "Predicted Y =", predict(x_value, m, c))


## 10. 🧠 Residual
\[
residual = Y-\hat{Y}
\]

For `X=1`, predicted Y is `2.8`, actual Y is `2`, so residual is `-0.8`.

This is the model asking: **How wrong was I on this observation?**


In [ ]:
for i in range(count):
    predicted_y = predict(x[i], m, c)
    residual = y[i] - predicted_y
    print("X =", x[i], "Actual =", y[i],
          "Predicted =", round(predicted_y, 2),
          "Residual =", round(residual, 2))


## 11. 🔢 Why square the errors?
If we simply sum residuals, positive and negative values can cancel.

Ordinary least squares uses:
\[
SSE=\sum(Y-\hat{Y})^2
\]

For this example:
\[
SSE=2.4
\]

The best-fit line is the line whose parameters minimize this total squared error.


In [ ]:
sse = 0

for i in range(count):
    predicted_y = predict(x[i], m, c)
    residual = y[i] - predicted_y
    squared_error = residual ** 2
    sse = sse + squared_error

print("SSE =", round(sse, 4))


## 12. 🔢 SSE, MSE, RMSE, MAE
- **SSE** = sum of squared errors
- **MSE** = average squared error
- **RMSE** = square root of MSE; returns roughly to Y's units
- **MAE** = average absolute error

For this data:
\[
SSE=2.4,\quad MSE=0.48,\quad RMSE\approx0.693
\]


In [ ]:
mse = sse / count
rmse = mse ** 0.5

absolute_error_total = 0
for i in range(count):
    absolute_error_total += abs(y[i] - predict(x[i], m, c))
mae = absolute_error_total / count

print("SSE =", round(sse, 4))
print("MSE =", round(mse, 4))
print("RMSE =", round(rmse, 4))
print("MAE =", round(mae, 4))


## 13. 📊 Visualization
The goal is to see **actual observations vs the learned line**, not just create a pretty chart.


In [ ]:
import matplotlib.pyplot as plt

predicted_values = [predict(v, m, c) for v in x]

plt.scatter(x, y, label="Actual")
plt.plot(x, predicted_values, label="Regression line")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()


## 14. ⚡ scikit-learn version
Mental translation of `fit()`:

`training data → find parameters → minimize error → store learned coefficient/intercept`

For simple linear regression, the learned parameters are `m` and `c`.


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array(x).reshape(-1, 1)
Y = np.array(y)

model = LinearRegression()
model.fit(X, Y)

print("sklearn slope =", model.coef_[0])
print("sklearn intercept =", model.intercept_)
print("predictions =", model.predict(X))


## 15. 🏗️ Data-engineering connection
Think in **two grains**.

**Row grain:** X, Y, deviations, prediction, residual, squared error.

**Dataset grain:** means, accumulated numerator/denominator, learned parameters, metrics.

Math `Σ` often maps naturally to Python running totals or SQL `SUM()`. Mean maps to SQL `AVG()`.


## 16. 💊 Pharma lens
Use public/synthetic data only.

Examples:
- historical product activity → future continuous performance
- marketing activity → sales response
- territory characteristics → numerical outcome

Reusable pattern: **features → continuous target**.


## 17. 🌱 PlantMind lens
Synthetic examples:
- sensor stress → degradation score
- vibration metric → health indicator
- physics-derived feature → continuous remaining-health proxy

Same ML concept, different domain. This is useful proof that the engineering skill is transferable.


## 18. ⚠️ Corrections captured from the course
1. In ML, think of an independent variable as **feature/input/predictor**; it does not mean nothing else can influence it.
2. Regression does not automatically prove **causation**.
3. `c` is specifically the **intercept**.
4. For this dataset, \(\sum X^2=55\); `86` is \(\sum Y^2\).
5. Deviations used to derive slope are relative to the **means**, not predicted values.
6. Prediction comes after training.


## 19. 🧪 Exercises
### Rebuild
Without looking above, recreate:
1. means
2. deviations
3. slope
4. intercept
5. `predict()`
6. SSE/MSE/RMSE

### Break it
Change one Y value. Predict first what will happen to:
- mean Y
- slope
- intercept
- SSE

Then run and compare.


## 20. ✅ Explain-back checkpoint
You have mastered this lesson when you can explain:

> Simple linear regression learns a slope and intercept that define a line. It chooses those parameters to minimize squared differences between actual and predicted values. `fit()` automates that learning step.

And you can rebuild the tiny example with basic Python.


## 21. ❓ Questions / gaps
Record anything that still feels magical:
- Why does least squares use squared errors?
- Can I explain training vs prediction?
- Can I explain SSE vs MSE vs RMSE?
- Can I translate sigma notation into loops and SQL?
- Can I explain why the line passes through `(x_mean, y_mean)`?
